In [2]:
!pip install -q numpy pandas seaborn matplotlib scikit-learn opfython xgboost scipy


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
# Standard library
import os
import json
import logging
import warnings

# Data manipulation & visualization
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Preprocessing
from sklearn.preprocessing import MinMaxScaler

# Model selection
from sklearn.model_selection import StratifiedKFold, ParameterGrid

# Classifiers
from sklearn.dummy import DummyClassifier
from opfython.models import SupervisedOPF
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC as SupportVectorMachineClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

# Metrics & evaluation
from sklearn.metrics import roc_curve, precision_recall_curve
from sklearn.metrics import balanced_accuracy_score, matthews_corrcoef
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score

# Statistics
from scipy.stats import wilcoxon
import scipy.stats as st

# Output directories
os.makedirs('../images', exist_ok=True)
os.makedirs('../outputs', exist_ok=True)

In [52]:
# Set default figure size and font sizes for plots
plt.rcParams.update({
    'axes.titlesize': 20,
    'axes.labelsize': 18,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'font.size': 24
})

In [6]:
df = pd.read_csv('../data/NPHA-doctor-visits.csv')
df

,Number of Doctors Visited,Age,Phyiscal Health,Mental Health,Dental Health,Employment,Stress Keeps Patient from Sleeping,Medication Keeps Patient from Sleeping,Pain Keeps Patient from Sleeping,Bathroom Needs Keeps Patient from Sleeping,Uknown Keeps Patient from Sleeping,Trouble Sleeping,Prescription Sleep Medication,Race,Gender
0,3,2,4,3,3,3,0,0,0,0,1,2,3,1,2
1,2,2,4,2,3,3,1,0,0,1,0,3,3,1,1
2,3,2,3,2,3,3,0,0,0,0,1,3,3,4,1
3,1,2,3,2,3,3,0,0,0,1,0,3,3,4,2
4,3,2,3,3,3,3,1,0,0,0,0,2,3,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
709,2,2,2,2,2,3,0,0,0,1,0,3,3,1,1
710,3,2,2,2,2,2,1,0,0,0,1,2,3,1,2
711,3,2,4,2,3,3,0,0,0,0,0,3,3,1,1
712,3,2,3,1,3,3,1,0,1,1,1,3,3,1,2


In [96]:
# Combine classes 2 and 3 into a single class (2) for the target variable
df['Number of Doctors Visited'] = df['Number of Doctors Visited'].replace(3, 2)

In [97]:
y = df['Number of Doctors Visited'].map({1: 0, 2: 1}).to_numpy()

feature_cols = df.columns.drop('Number of Doctors Visited').tolist()
X_df = pd.get_dummies(df[feature_cols], columns=feature_cols, drop_first=False)
feature_names = X_df.columns.to_numpy()

X = X_df.values

In [101]:
SEED = 42
N_SPLITS = 10
outer_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

In [ ]:
param_grids = {
    'KNN':   {'n_neighbors': [10, 20, 30, 40], 'metric': ['euclidean', 'manhattan', 'minkowski', 'cosine']},
    'DT':    {'max_depth': [3, 5, 7], 'min_samples_split': [10, 20, 30], 'min_samples_leaf': [5, 10]},
    'RF':    {'n_estimators': [100, 200, 300], 'max_depth': [3, 5, 7], 'min_samples_leaf': [2, 5, 10]},
    'SVM':   {'C': [0.1, 1, 10], 'kernel': ['rbf'], 'gamma': ['scale', 0.1], 'probability': [True]},
    'MLP':   {'hidden_layer_sizes': [(32, 16), (64,)], 'alpha': [0.01, 0.1, 1.0], 'learning_rate_init': [0.001], 'activation': ['relu', 'tanh'], 'solver': ['adam'], 'max_iter': [2000], 'early_stopping': [True], 'random_state': [SEED]},
    'LR':    {'C': [0.01, 0.1, 1, 10], 'solver': ['saga'], 'penalty': ['l1', 'l2']},
    'XGB':   {'n_estimators': [100, 200], 'max_depth': [3, 5], 'learning_rate': [0.01, 0.05, 0.1], 'subsample': [0.8], 'colsample_bytree': [0.8], 'reg_alpha': [0.1, 1.0]},
    'NB':    {'var_smoothing': [1e-11, 1e-10, 1e-9, 1e-8, 1e-7]},
    'NC':    {'metric': ['euclidean', 'manhattan'], 'shrink_threshold': [None, 0.1, 0.5, 1.0]},
    'OPF':   {'distance': ['euclidean', 'squared_euclidean', 'log_squared_euclidean', 'manhattan', 'canberra', 'chebyshev']},
    'DUMMY': {'strategy': ['most_frequent']},
}
model_names = list(param_grids.keys())

In [ ]:
def build_estimator(name, params):
    if name == 'KNN':   return KNeighborsClassifier(**params)
    if name == 'DT':    return DecisionTreeClassifier(random_state=SEED, **params)
    if name == 'RF':    return RandomForestClassifier(random_state=SEED, n_jobs=-1, **params)
    if name == 'SVM':   return SupportVectorMachineClassifier(random_state=SEED, **params)
    if name == 'MLP':   return MLPClassifier(**params)
    if name == 'LR':    return LogisticRegression(max_iter=1000, random_state=SEED, **params)
    if name == 'XGB':   return XGBClassifier(random_state=SEED, n_jobs=-1, eval_metric='logloss', **params)
    if name == 'NB':    return GaussianNB(**params)
    if name == 'NC':    return NearestCentroid(**params)
    if name == 'OPF':   return SupervisedOPF(**params)
    if name == 'DUMMY': return DummyClassifier(random_state=SEED, **params)
    raise ValueError(name)

def fit_predict(name, model, X_tr, y_tr, X_ev):
    if name == 'OPF':
        model.fit(np.array(X_tr), np.array(y_tr, dtype=np.int32))
        return model, model.predict(np.array(X_ev))
    model.fit(X_tr, y_tr)
    return model, model.predict(X_ev)

def score_proba(name, model, X_ev):
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X_ev)
        classes = list(model.classes_)
        return proba[:, classes.index(1)], True
    return None, False

In [104]:
def validation_metric(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro')

In [105]:
def select_mdi_features(X_train, y_train, coverage=0.80, seed=SEED):
    selector = RandomForestClassifier(n_estimators=500, class_weight='balanced', random_state=seed, n_jobs=-1)
    selector.fit(X_train, y_train)
    importances = selector.feature_importances_
    order = np.argsort(importances)[::-1]
    cumulative = np.cumsum(importances[order])
    n_selected = np.searchsorted(cumulative, coverage) + 1
    mask = np.zeros(len(importances), dtype=bool)
    mask[order[:n_selected]] = True
    return mask, importances

In [ ]:
metric_names = [
    'acc', 'balanced_acc', 'mcc',
    'precision', 'recall', 'f1', 'auroc',
    'precision_cls0', 'recall_cls0', 'f1_cls0',
    'precision_cls1', 'recall_cls1', 'f1_cls1',
]
all_names = model_names + ['VOTING']

results           = {name: {m: [] for m in metric_names} for name in all_names}
best_params_per_fold = {name: [] for name in model_names}
confusion_matrices   = {name: [] for name in all_names}

mdi_masks       = {name: [] for name in model_names}
mdi_importances = {name: [] for name in model_names}
top3_per_fold   = []

oof_true     = np.full(len(y), -1, dtype=int)
oof_pred     = {name: np.full(len(y), -1, dtype=int) for name in all_names}
oof_score    = {name: np.full(len(y), np.nan) for name in all_names}
oof_fold_id  = np.full(len(y), -1, dtype=int)

logging.getLogger("opfython").disabled = True
logging.disable(logging.CRITICAL)
warnings.filterwarnings("ignore")

In [ ]:
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
    X_train_outer, X_test_outer = X[train_idx], X[test_idx]
    y_train_outer, y_test_outer = y[train_idx], y[test_idx]

    mdi_mask, mdi_imp = select_mdi_features(X_train_outer, y_train_outer, coverage=0.80)
    X_train_mdi = X_train_outer[:, mdi_mask]
    X_test_mdi  = X_test_outer[:, mdi_mask]

    inner_splits = list(inner_cv.split(X_train_outer, y_train_outer))
    inner_mdi_masks = []
    for tr_in_idx, val_in_idx in inner_splits:
        mask_in, _ = select_mdi_features(X_train_outer[tr_in_idx], y_train_outer[tr_in_idx], coverage=0.80)
        inner_mdi_masks.append(mask_in)

    f1_val_by_model = {}
    fold_test_preds = {}

    for name in model_names:
        mdi_masks[name].append(mdi_mask)
        mdi_importances[name].append(pd.Series(mdi_imp, index=feature_names))

        best_f1, best_params = -np.inf, None
        for params in ParameterGrid(param_grids[name]):
            inner_f1s = []
            for inner_i, (tr_in_idx, val_in_idx) in enumerate(inner_splits):
                mask_in = inner_mdi_masks[inner_i]

                X_tr_in_raw  = X_train_outer[tr_in_idx][:, mask_in]
                X_val_in_raw = X_train_outer[val_in_idx][:, mask_in]
                y_tr_in      = y_train_outer[tr_in_idx]
                y_val_in     = y_train_outer[val_in_idx]

                scaler_in  = MinMaxScaler()
                X_tr_in_s  = scaler_in.fit_transform(X_tr_in_raw)
                X_val_in_s = scaler_in.transform(X_val_in_raw)

                estimator = build_estimator(name, params)
                try:
                    _, y_pred_val = fit_predict(name, estimator, X_tr_in_s, y_tr_in, X_val_in_s)
                    inner_f1s.append(validation_metric(y_val_in, y_pred_val))
                except Exception:
                    inner_f1s.append(-np.inf)

            mean_f1 = np.mean(inner_f1s) if inner_f1s else -np.inf
            if mean_f1 > best_f1:
                best_f1, best_params = mean_f1, params

        best_params_per_fold[name].append(best_params)
        f1_val_by_model[name] = best_f1

        scaler_out     = MinMaxScaler()
        X_train_scaled = scaler_out.fit_transform(X_train_mdi)
        X_test_scaled  = scaler_out.transform(X_test_mdi)

        final_model = build_estimator(name, best_params)
        final_model, y_pred_test = fit_predict(name, final_model, X_train_scaled, y_train_outer, X_test_scaled)

        y_score, has_proba = score_proba(name, final_model, X_test_scaled)
        auroc = roc_auc_score(y_test_outer, y_score) if has_proba else np.nan

        results[name]['acc'].append(accuracy_score(y_test_outer, y_pred_test))
        results[name]['balanced_acc'].append(balanced_accuracy_score(y_test_outer, y_pred_test))
        results[name]['mcc'].append(matthews_corrcoef(y_test_outer, y_pred_test))
        results[name]['precision'].append(precision_score(y_test_outer, y_pred_test, average='macro', zero_division=0))
        results[name]['recall'].append(recall_score(y_test_outer, y_pred_test, average='macro', zero_division=0))
        results[name]['f1'].append(f1_score(y_test_outer, y_pred_test, average='macro', zero_division=0))
        results[name]['auroc'].append(auroc)
        for cls in [0, 1]:
            results[name][f'precision_cls{cls}'].append(
                precision_score(y_test_outer, y_pred_test, pos_label=cls, average='binary', zero_division=0))
            results[name][f'recall_cls{cls}'].append(
                recall_score(y_test_outer, y_pred_test, pos_label=cls, average='binary', zero_division=0))
            results[name][f'f1_cls{cls}'].append(
                f1_score(y_test_outer, y_pred_test, pos_label=cls, average='binary', zero_division=0))
        confusion_matrices[name].append(confusion_matrix(y_test_outer, y_pred_test, labels=[0, 1]))

        oof_pred[name][test_idx]  = y_pred_test
        oof_score[name][test_idx] = y_score if has_proba else np.nan
        fold_test_preds[name]     = y_pred_test

    top3 = sorted(f1_val_by_model, key=f1_val_by_model.get, reverse=True)[:3]
    top3_per_fold.append(top3)

    stacked_preds = np.stack([fold_test_preds[name] for name in top3], axis=1)
    y_pred_voting = st.mode(stacked_preds, axis=1, keepdims=False).mode

    results['VOTING']['acc'].append(accuracy_score(y_test_outer, y_pred_voting))
    results['VOTING']['balanced_acc'].append(balanced_accuracy_score(y_test_outer, y_pred_voting))
    results['VOTING']['mcc'].append(matthews_corrcoef(y_test_outer, y_pred_voting))
    results['VOTING']['precision'].append(precision_score(y_test_outer, y_pred_voting, average='macro', zero_division=0))
    results['VOTING']['recall'].append(recall_score(y_test_outer, y_pred_voting, average='macro', zero_division=0))
    results['VOTING']['f1'].append(f1_score(y_test_outer, y_pred_voting, average='macro', zero_division=0))
    results['VOTING']['auroc'].append(np.nan)
    for cls in [0, 1]:
        results['VOTING'][f'precision_cls{cls}'].append(
            precision_score(y_test_outer, y_pred_voting, pos_label=cls, average='binary', zero_division=0))
        results['VOTING'][f'recall_cls{cls}'].append(
            recall_score(y_test_outer, y_pred_voting, pos_label=cls, average='binary', zero_division=0))
        results['VOTING'][f'f1_cls{cls}'].append(
            f1_score(y_test_outer, y_pred_voting, pos_label=cls, average='binary', zero_division=0))
    confusion_matrices['VOTING'].append(confusion_matrix(y_test_outer, y_pred_voting, labels=[0, 1]))

    oof_true[test_idx]           = y_test_outer
    oof_pred['VOTING'][test_idx] = y_pred_voting
    oof_fold_id[test_idx]        = fold

In [ ]:
mdi_importances_df = {}
mdi_ranking = {}
mdi_selection_frequency = {}

for name in model_names:
    imp_df = pd.concat(mdi_importances[name], axis=1)
    imp_df.columns = [f'fold_{i+1}' for i in range(len(mdi_importances[name]))]
    mdi_importances_df[name] = imp_df
    mdi_ranking[name] = imp_df.mean(axis=1).sort_values(ascending=False)
    mdi_selection_frequency[name] = pd.Series(mdi_masks[name]).apply(
        lambda m: pd.Series(feature_names[m])
    ).stack().value_counts()

In [ ]:
summary_rows = []
for name in all_names:
    row = {'model': name}
    for m in metric_names:
        vals = results[name][m]
        row[f'{m}_mean'] = np.mean(vals)
        row[f'{m}_std'] = np.std(vals)
    summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows).set_index('model')
summary_df

In [ ]:
def confidence_interval(data, confidence=0.95):
    data = np.array(data)
    n = len(data)
    if n <= 1: return 0.0
    se = st.sem(data)
    h = se * st.t.ppf((1 + confidence) / 2., n - 1)
    return h if not np.isnan(h) else 0.0

for metric in metric_names:
    df_metric = pd.DataFrame({name: results[name][metric] for name in all_names}, index=[f'Fold {i+1}' for i in range(N_SPLITS)])
    means = df_metric.mean()
    cis = df_metric.apply(lambda x: confidence_interval(x.dropna()))
    min_y = max(0, (means - cis).min() - 0.05)

    fig, axes = plt.subplots(1, 2, figsize=(20, 6))
    for name in df_metric.columns:
        axes[0].plot(df_metric.index, df_metric[name], marker='o', label=name, linewidth=2)
    axes[0].set_title(f'Per-fold {metric}')
    axes[0].set_xlabel('Fold')
    axes[0].set_ylabel(metric)
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.4)
    axes[0].set_ylim(min_y, 1.0)
    axes[0].set_xticklabels(df_metric.index, rotation=45, ha='right')

    bars = axes[1].bar(means.index, means.values, yerr=cis.values, capsize=5, color=sns.color_palette('viridis', len(means)), alpha=0.85)
    for bar, mean, ci in zip(bars, means.values, cis.values):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + ci + 0.01, f'{mean:.3f}', ha='center', fontsize=9, fontweight='bold')
    axes[1].set_title(f'Mean 95% CI - {metric}')
    axes[1].set_xlabel('Model')
    axes[1].set_ylabel(f'Mean {metric}')
    axes[1].grid(True, alpha=0.4, axis='y')
    axes[1].set_ylim(min_y, 1.0)
    axes[1].set_xticklabels(means.index, rotation=45, ha='right', fontsize=11)

    plt.tight_layout()
    plt.savefig(f'../images/perf_ci_{metric}.pdf', format='pdf', bbox_inches='tight')
    plt.show()

In [ ]:
alpha = 0.05

for metric in metric_names:
    pairs = [(m1, m2) for i, m1 in enumerate(all_names) for m2 in all_names[i+1:]]
    raw_p   = []
    w_stats = []

    for m1, m2 in pairs:
        s1 = [v for v in results[m1][metric] if not np.isnan(v)]
        s2 = [v for v in results[m2][metric] if not np.isnan(v)]
        if len(s1) < 2 or len(s1) != len(s2):
            raw_p.append(np.nan); w_stats.append(np.nan)
            continue
        try:
            stat, p = wilcoxon(s1, s2, zero_method='zsplit', correction=False)
            raw_p.append(p); w_stats.append(stat)
        except ValueError:
            raw_p.append(np.nan); w_stats.append(np.nan)

    raw_p_arr = np.array(raw_p, dtype=float)
    adj_p     = np.full(len(raw_p), np.nan)
    valid_idx = np.where(~np.isnan(raw_p_arr))[0]
    if len(valid_idx) > 0:
        valid_pvals  = raw_p_arr[valid_idx]
        n            = len(valid_pvals)
        sorted_order = np.argsort(valid_pvals)
        holm = np.zeros(n)
        for rank, idx in enumerate(sorted_order):
            holm[idx] = min(valid_pvals[idx] * (n - rank), 1.0)
        for rank in range(1, n):
            holm[sorted_order[rank]] = max(holm[sorted_order[rank]], holm[sorted_order[rank - 1]])
        adj_p[valid_idx] = holm

    print(f'\n=== {metric.upper()} (Wilcoxon signed-rank + Holm) ===')
    for i, (m1, m2) in enumerate(pairs):
        print(f'\n{m1} vs {m2}:')
        if np.isnan(raw_p[i]):
            print('\t- Teste não pôde ser computado.')
            continue
        mean1 = np.nanmean(results[m1][metric])
        mean2 = np.nanmean(results[m2][metric])
        print(f'\t- Média {m1}: {mean1:.4f} | Média {m2}: {mean2:.4f}')
        print(f'\t- W: {w_stats[i]:.4f} | p bruto: {raw_p[i]:.4f} | p ajustado (Holm): {adj_p[i]:.4f}')
        if adj_p[i] < alpha:
            best = m1 if mean1 > mean2 else m2
            print(f'\t-> Diferença significativa (p_adj < {alpha}). Melhor: {best}')
        else:
            print(f'\t-> Sem diferença significativa (p_adj >= {alpha}).')

In [ ]:
metrics_rows = []
for name in all_names:
    for fold_idx in range(N_SPLITS):
        row = {'model': name, 'fold': fold_idx + 1}
        for m in metric_names:
            row[m] = results[name][m][fold_idx]
        metrics_rows.append(row)

metrics_per_fold_df = pd.DataFrame(metrics_rows)
metrics_per_fold_df.to_csv('../outputs/metrics_per_fold.csv', index=False)

In [ ]:
import json

config = {
    'seed': SEED,
    'n_splits': N_SPLITS,
    'target_mapping': {1: 0, 2: 1},
    'mdi_coverage': 0.80,
    'param_grids': param_grids,
    'best_params_per_fold': best_params_per_fold,
    'top3_per_fold': top3_per_fold,
}

with open('../outputs/config.json', 'w') as f:
    json.dump(config, f, indent=2, default=str)

In [ ]:
oof_rows = {'sample_index': np.arange(len(y)), 'fold': oof_fold_id, 'y_true': oof_true}
for name in all_names:
    oof_rows[f'{name}_pred'] = oof_pred[name]
    oof_rows[f'{name}_score'] = oof_score[name]

oof_df = pd.DataFrame(oof_rows)
oof_df.to_csv('../outputs/oof_predictions.csv', index=False)

In [ ]:
np.savez('../outputs/confusion_matrices.npz', **{name: np.array(confusion_matrices[name]) for name in all_names})

cm_rows = []
for name in all_names:
    for fold_idx, cm in enumerate(confusion_matrices[name], start=1):
        cm_rows.append({'model': name, 'fold': fold_idx, 'tn': cm[0, 0], 'fp': cm[0, 1], 'fn': cm[1, 0], 'tp': cm[1, 1]})
cm_df = pd.DataFrame(cm_rows)
cm_df.to_csv('../outputs/confusion_matrices.csv', index=False)

In [ ]:
class_labels = ['0-1 Visit', '2+ Visits']
cm_sums = {name: np.sum(confusion_matrices[name], axis=0) for name in all_names}
n_models = len(all_names)
n_cols = 4
n_rows = int(np.ceil(n_models / n_cols))
plt.figure(figsize=(5 * n_cols, 4 * n_rows))
for j, name in enumerate(all_names):
    plt.subplot(n_rows, n_cols, j + 1)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm_sums[name], display_labels=class_labels)
    disp.plot(cmap='Blues', ax=plt.gca(), values_format='d', colorbar=False)
    plt.title(name, fontsize=12, fontweight='bold')
    plt.xlabel('Predicted', fontsize=9)
    plt.ylabel('True', fontsize=9)
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.yticks(fontsize=8)
plt.suptitle('Aggregated Confusion Matrices (sum over folds)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.subplots_adjust(top=0.93)
plt.savefig('../images/confusion_matrices_grid.pdf', format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
for name in model_names:
    mdi_importances_df[name].to_csv(f'../outputs/mdi_importances_per_fold_{name}.csv')
    mdi_ranking[name].to_csv(f'../outputs/mdi_ranking_mean_{name}.csv', header=['mean_importance'])
    mdi_selection_frequency[name].to_csv(f'../outputs/mdi_selection_frequency_{name}.csv', header=['n_folds_selected'])

In [ ]:
voting_rows = []
for fold_idx in range(N_SPLITS):
    row = {'fold': fold_idx + 1, 'top3_models': top3_per_fold[fold_idx]}
    for m in metric_names:
        row[m] = results['VOTING'][m][fold_idx]
    voting_rows.append(row)
voting_df = pd.DataFrame(voting_rows)
voting_df.to_csv('../outputs/voting_results.csv', index=False)

In [ ]:
plt.figure(figsize=(8, 6))
for name in all_names:
    valid = ~np.isnan(oof_score[name])
    if valid.sum() == 0:
        continue
    fpr, tpr, _ = roc_curve(oof_true[valid], oof_score[name][valid])
    plt.plot(fpr, tpr, label=name)
plt.plot([0, 1], [0, 1], linestyle='--', color='grey')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(fontsize=9)
plt.savefig('../images/roc_curves.pdf', format='pdf', bbox_inches='tight')
plt.show()

plt.figure(figsize=(8, 6))
for name in all_names:
    valid = ~np.isnan(oof_score[name])
    if valid.sum() == 0:
        continue
    precision, recall, _ = precision_recall_curve(oof_true[valid], oof_score[name][valid])
    plt.plot(recall, precision, label=name)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend(fontsize=9)
plt.savefig('../images/pr_curves.pdf', format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
summary_df.to_csv('../outputs/summary_metrics.csv')
summary_df.to_latex('../outputs/summary_metrics.tex')